# Final Capstone — Personal Learning Coach

You will compose **every module** into a single product: a coach that helps a student progress through this very course.

## Architecture
```
              ┌────────── MCP server (Module 4) ──────────┐
              │   resources: courses, user progress       │
              │   tools: enrol, mark_complete, recommend  │
              └───────────────────┬───────────────────────┘
                                  │
┌─────────────┐   classify   ┌────▼─────┐   plan   ┌──────────────┐
│ user query  ├─────────────▶│  Coach   │─────────▶│  LangGraph   │
└─────────────┘  fine-tuned  │  agent   │  (M5)    │ multi-agent  │
                  small LLM  └────┬─────┘          └──────┬───────┘
                  (Module 2)      │ retrieve              │ act
                                  ▼                       ▼
                       ┌────────────────────┐   ┌──────────────────┐
                       │ RAG over course    │   │  MCP tool calls  │
                       │ content (Module 3) │   │   (Module 4)     │
                       └────────────────────┘   └──────────────────┘
                                  │
                                  ▼
                       ┌────────────────────┐
                       │  Observability:    │
                       │  traces, evals,    │
                       │  budgets, alerts   │
                       └────────────────────┘
```

This is where every concept stops being academic.

## Required components — checklist

| From | What must appear |
|---|---|
| Module 1 | Strict structured outputs at every LLM boundary; eval harness in CI |
| Module 2 | A small fine-tuned classifier (intent, difficulty, or content-tag) — quantized |
| Module 3 | RAG over the courses own notebooks/PDFs, hybrid + reranked, with citations |
| Module 4 | An MCP server exposing `progress`, `enrol`, `recommend`, `complete_lesson` |
| Module 5 | LangGraph agent: `Coach → Tutor → Quizzer → Reflector` with HITL on low-confidence answers |
| Cross-cutting | Budget cap, audit log, eval pipeline that runs nightly |


## Phase 0 — Architecture write-up (20 pts)

Before writing code, submit a 1-page architecture document covering:
1. Component diagram (the one above, adapted to your choices).
2. Data flow for one user query, end-to-end.
3. Tradeoff table: 3 alternatives you considered and rejected, with the reason.
4. Service-Level Objectives (SLOs): p95 latency, $/session, success rate.

*(This is graded BEFORE you start coding. It saves you from rebuilding.)*

In [ ]:
# Phase 1 — Skeleton

# This is intentionally just stubs. The capstone is to fill them with the code
# you wrote in modules 1–5, glued together.

from typing import TypedDict, List

class CoachState(TypedDict):
    user_id: str
    query: str
    intent: str            # from your fine-tuned classifier (M2)
    retrieved: List[dict]  # from RAG (M3)
    actions: List[dict]    # MCP tool calls executed (M4)
    answer: str
    cost_usd: float
    trace_id: str

def classify_intent(query: str) -> str:
    # TODO 1: call your quantized fine-tuned model from Module 2.
    raise NotImplementedError

def rag_retrieve(query: str) -> List[dict]:
    # TODO 2: reuse the hybrid+rerank retriever from Module 3.
    raise NotImplementedError

def mcp_call(tool: str, args: dict) -> dict:
    # TODO 3: speak MCP to your server from Module 4.
    raise NotImplementedError


In [ ]:
# Phase 2 — The graph

from langgraph.graph import StateGraph, END

def coach(state: CoachState) -> dict:
    # Plan: decide which of {tutor, quizzer, reflector, mcp_action} to invoke.
    # TODO 4
    raise NotImplementedError

def tutor(state: CoachState) -> dict:
    # Answer pedagogically using retrieved context. Cite sources.
    # TODO 5
    raise NotImplementedError

def quizzer(state: CoachState) -> dict:
    # Generate a check-your-understanding question; grade the users reply.
    # TODO 6
    raise NotImplementedError

def reflector(state: CoachState) -> dict:
    # Summarise what was covered, log progress via MCP, recommend next lesson.
    # TODO 7
    raise NotImplementedError

# TODO 8: assemble the StateGraph; add HITL interrupt when reflector confidence < 0.6.


In [ ]:
# Phase 3 — Continuous evaluation pipeline

# A frozen test set of 50 student interactions with gold expectations.
# This runs nightly; PRs that regress >2% on any metric are blocked.

NIGHTLY_EVAL = [
    # {"user": "...", "query": "...", "expects_intent": "...", "expects_cite": [...], "expects_action": None | "..."}
]

def nightly():
    rows = []
    for item in NIGHTLY_EVAL:
        # TODO 9: invoke graph, score on intent / faithfulness / action / cost / latency
        pass
    # TODO 10: write rows to a dashboard (Grafana, Langfuse, or a JSON+chart) and alert on regression.


## Phase 4 — Production monitoring stack (REQUIRED)

You must wire and demo:
- **Tracing**: every LLM + tool call exported (Langfuse / OTLP).
- **Metrics**: tokens, $, p50/p95 latency, error rate by stage — Prometheus + Grafana **or** an equivalent dashboard.
- **Eval drift**: nightly Ragas + custom-judge runs; >2% regression → alert.
- **Budget guardrails**: per-user `$/day` ceiling; hard-cutoff with a polite refusal.
- **Feedback loop**: thumbs UI → weekly batch into the eval set.

Submit screenshots of the live dashboard with traffic on it.

## Submission rubric — out of 200

| Category | Pts |
|---|---|
| Architecture write-up + tradeoff analysis | 20 |
| Functional completeness — all 5 modules represented | 40 |
| End-to-end task success rate ≥ 0.7 (user study or sim) | 40 |
| Cost & latency SLOs documented and met | 25 |
| Observability dashboard (traces, metrics, evals, $) | 25 |
| Safety: prompt-injection, PII, refusal handling | 20 |
| Continuous-eval pipeline (regression on every PR / nightly) | 20 |
| Live demo + Q&A | 10 |

**≥ 150 = distinction.** This is the artefact you put on your resume.